In [1]:
!pip install -q pandas scikit-learn

In [3]:
!ls

online_shoppers_intention.csv


In [16]:
!python /content/feature_engineering.py /content/online_shoppers_intention.csv

Rows/columns: (12330, 18); missing cells: 0; exact duplicates: 125

Confusion matrix [ [TN, FP], [FN, TP] ]:
[[1285  799]
 [  98  284]]

Class report:
              precision    recall  f1-score   support

 no purchase       0.93      0.62      0.74      2084
    purchase       0.26      0.74      0.39       382

    accuracy                           0.64      2466
   macro avg       0.60      0.68      0.56      2466
weighted avg       0.83      0.64      0.69      2466

ROC-AUC: 0.7502
Average precision: 0.3295

Scores are from a random stratified split, not a time-based future holdout.
Compare this run with a raw-feature baseline before concluding engineering helped.


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("/content/online_shoppers_intention.csv")

y = df["Revenue"].astype(int)
X = df.drop(columns="Revenue")

# Exclude PageValues from both models for a fair comparison
X = X.drop(columns="PageValues")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Purchase rate in test set:", round(y_test.mean(), 3))

Training rows: 9864
Test rows: 2466
Purchase rate in test set: 0.155


Prepare the raw-feature baseline

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

category_cols = [
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "Month",
    "VisitorType",
    "Weekend",
]

numeric_cols = [column for column in X_train.columns if column not in category_cols]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("fill_missing", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
            ]),
            numeric_cols,
        ),
        (
            "categories",
            Pipeline([
                ("fill_missing", SimpleImputer(strategy="most_frequent")),
                ("one_hot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            category_cols,
        ),
    ]
)

baseline_model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
    )),
])

baseline_model.fit(X_train, y_train)

print("Raw-feature baseline is trained.")

Raw-feature baseline is trained.


Evaluate the raw-feature baseline

In [11]:
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

baseline_predictions = baseline_model.predict(X_test)
baseline_probabilities = baseline_model.predict_proba(X_test)[:, 1]

print("Confusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_test, baseline_predictions))

print("\nClassification report:")
print(classification_report(
    y_test,
    baseline_predictions,
    target_names=["no purchase", "purchase"],
    zero_division=0,
))

print("ROC-AUC:", round(roc_auc_score(y_test, baseline_probabilities), 4))
print("Average precision:", round(
    average_precision_score(y_test, baseline_probabilities), 4
))

Confusion matrix [ [TN, FP], [FN, TP] ]:
[[1267  817]
 [  96  286]]

Classification report:
              precision    recall  f1-score   support

 no purchase       0.93      0.61      0.74      2084
    purchase       0.26      0.75      0.39       382

    accuracy                           0.63      2466
   macro avg       0.59      0.68      0.56      2466
weighted avg       0.83      0.63      0.68      2466

ROC-AUC: 0.7348
Average precision: 0.3211


Add the engineered features

In [12]:
import numpy as np

def add_behavior_features(data):
    data = data.copy()

    page_groups = [
        ("Administrative", "Administrative_Duration"),
        ("Informational", "Informational_Duration"),
        ("ProductRelated", "ProductRelated_Duration"),
    ]

    for page_count, duration in page_groups:
        data[f"{page_count}_visited"] = (data[page_count] > 0).astype(int)

        data[f"{page_count}_seconds_per_page"] = np.divide(
            data[duration],
            data[page_count],
            out=np.zeros(len(data), dtype=float),
            where=data[page_count].to_numpy() > 0,
        )

    data["total_pages"] = data[
        ["Administrative", "Informational", "ProductRelated"]
    ].sum(axis=1)

    data["total_duration"] = data[
        [
            "Administrative_Duration",
            "Informational_Duration",
            "ProductRelated_Duration",
        ]
    ].sum(axis=1)

    month_number = data["Month"].map({
        "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
        "May": 5, "June": 6, "Jul": 7, "Aug": 8,
        "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12,
    })

    data["month_sin"] = np.sin(2 * np.pi * month_number / 12)
    data["month_cos"] = np.cos(2 * np.pi * month_number / 12)

    return data

X_train_engineered = add_behavior_features(X_train)
X_test_engineered = add_behavior_features(X_test)

print("Training columns before:", X_train.shape[1])
print("Training columns after:", X_train_engineered.shape[1])

Training columns before: 16
Training columns after: 26


Train the engineered-feature model

In [13]:
engineered_category_cols = [
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "Month",
    "VisitorType",
    "Weekend",
]

engineered_numeric_cols = [
    column for column in X_train_engineered.columns
    if column not in engineered_category_cols
]

engineered_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("fill_missing", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
            ]),
            engineered_numeric_cols,
        ),
        (
            "categories",
            Pipeline([
                ("fill_missing", SimpleImputer(strategy="most_frequent")),
                ("one_hot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            engineered_category_cols,
        ),
    ]
)

engineered_model = Pipeline([
    ("preprocess", engineered_preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
    )),
])

engineered_model.fit(X_train_engineered, y_train)

print("Engineered-feature model is trained.")

Engineered-feature model is trained.


Evaluate the engineered-feature model

In [14]:
engineered_predictions = engineered_model.predict(X_test_engineered)
engineered_probabilities = engineered_model.predict_proba(X_test_engineered)[:, 1]

print("Confusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_test, engineered_predictions))

print("\nClassification report:")
print(classification_report(
    y_test,
    engineered_predictions,
    target_names=["no purchase", "purchase"],
    zero_division=0,
))

print("ROC-AUC:", round(roc_auc_score(y_test, engineered_probabilities), 4))
print("Average precision:", round(
    average_precision_score(y_test, engineered_probabilities), 4
))

Confusion matrix [ [TN, FP], [FN, TP] ]:
[[1276  808]
 [  97  285]]

Classification report:
              precision    recall  f1-score   support

 no purchase       0.93      0.61      0.74      2084
    purchase       0.26      0.75      0.39       382

    accuracy                           0.63      2466
   macro avg       0.60      0.68      0.56      2466
weighted avg       0.83      0.63      0.68      2466

ROC-AUC: 0.741
Average precision: 0.3145


Compare both models

In [15]:
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

results = []

for name, model, test_data in [
    ("Raw features", baseline_model, X_test),
    ("Engineered features", engineered_model, X_test_engineered),
]:
    predictions = model.predict(test_data)
    probabilities = model.predict_proba(test_data)[:, 1]

    results.append({
        "Model": name,
        "Purchase precision": precision_score(y_test, predictions, zero_division=0),
        "Purchase recall": recall_score(y_test, predictions, zero_division=0),
        "Purchase F1": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "Average precision": average_precision_score(y_test, probabilities),
    })

comparison = pd.DataFrame(results).set_index("Model").round(3)
display(comparison)

,Purchase precision,Purchase recall,Purchase F1,ROC-AUC,Average precision
Model,,,,,
Raw features,0.259,0.749,0.385,0.735,0.321
Engineered features,0.261,0.746,0.386,0.741,0.314
